# Post-Training Diagnostic Analysis & Physical Stress Testing

This notebook evaluates the detection behavior, peak-counting accuracy, and physical resolution limits of the trained **`DenseDetector`** model.

### Key Sections:
1. **Model Loading & Grid Setup**: Ingests the model checkpoint and frequency axis configuration.
2. **Batch Peak Counting Accuracy**: Evaluates exact and $\pm 1$ peak count accuracy across synthetic spectra.
3. **Visual Inspection on Multi-Peak Spectra**: Overlays detections on randomly generated spectra ($N=5$ and $N=10$ peaks).
4. **Physical Stress Test 1 (Sandwiched Clusters)**: Evaluates localization when small peaks are trapped between large flanking peaks.
5. **Physical Stress Test 2 (Separation Limit)**: Tests the physical grid and NMS resolution limits as peak separation approaches linewidth $\gamma$.
6. **Failure Case Visualization**: Inspects specific failure profiles from the batch test.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os

# Add project root to sys.path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.signal_sample_module import (
    generate_dataset,
    sample_true_peaks,
    generate_sandwiched_cluster,
    generate_close_peaks_sample,
    convert_peaks_to_signal
)
from src.inference import load_model, predict_peaks_dense

# Set reproducibility seeds and plot styling
torch.manual_seed(0)
rng = np.random.default_rng(0)
plt.rcParams["figure.dpi"] = 110

# Hardware device selection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Load Model & Configurations

In [ ]:
model_path = '../saved_models/dense_model.pt'
config_path = '../saved_models/dense_model_config.json'

cnn_model, cfg, device = load_model(model_path, config_path, device=device, mode="eval")
W = cfg.W  # Frequency mesh grid (cm^-1)

print(f"Model loaded: {cfg.n_points} spectral points, frequency range: [{cfg.pos_range[0]}, {cfg.pos_range[1]}] cm^-1")

## 2. Batch Peak Counting Accuracy
Evaluates the model on $N_{\text{test}}$ synthetic spectra to assess whether the total detected peak count matches ground truth.

In [ ]:
N_TEST = 500
print(f"Generating and evaluating {N_TEST} test spectra...")

X_test, peaks_test = generate_dataset(N_TEST, rng, cfg)
X_test = torch.tensor(X_test, dtype=torch.float32)
X_test_raw = X_test[:, 0, :] if X_test.dim() > 2 else X_test

count_correct = 0
count_errors = []
bad_pred, bad_raw, bad_peaks = [], [], []

cnn_model.eval()
for b in range(N_TEST):
    n_true = len(peaks_test[b])
    detected = predict_peaks_dense(cnn_model, X_test_raw[b].numpy(), cfg=cfg, device=device)
    n_pred = len(detected)
    
    error = n_pred - n_true
    count_errors.append(error)
    
    if n_pred == n_true:
        count_correct += 1
    else:
        bad_pred.append(detected)
        bad_raw.append(X_test_raw[b].numpy())
        bad_peaks.append(peaks_test[b])

count_errors = np.array(count_errors)
within_one = np.sum(np.abs(count_errors) <= 1)

print("\n=== Peak Count Evaluation Summary ===")
print(f"Exact count match:    {100 * count_correct / N_TEST:.1f}% ({count_correct}/{N_TEST})")
print(f"Within +/- 1 peak:    {100 * within_one / N_TEST:.1f}% ({within_one}/{N_TEST})")
print(f"Mean count error:     {np.mean(count_errors):+.3f}")

## 3. Visual Detection Inspection on Random Multi-Peak Spectra
Overlays true peak positions (blue solid lines) and FCN detections (red dashed lines) on spectra with 5 and 10 peaks.

In [ ]:
target_counts = [5, 10]
fig, axes = plt.subplots(1, len(target_counts), figsize=(18, 4))

for ax, target_n in zip(axes, target_counts):
    # Sample a profile with exactly target_n peaks
    for _ in range(200):
        true_p = sample_true_peaks(rng, cfg, min_sep_factor=1.0, min_sep_offset=1.0)
        if len(true_p) == target_n:
            raw = convert_peaks_to_signal(true_p, cfg.W, peak_type='lorentzian')
            break
    
    detected = predict_peaks_dense(cnn_model, raw, cfg=cfg, device=device)

    ax.plot(W, raw, ".", ms=2, color="gray", label="Measurement")
    for A, pos, gamma in true_p:
        ax.axvline(pos, color="#2b6cb0", alpha=0.4, lw=1.5, label="True Peak" if pos == true_p[0][1] else "")
    for conf, A, pos, gamma in detected:
        ax.axvline(pos, color="#e53e3e", ls="--", lw=1.2, label="FCN Detected" if pos == detected[0][2] else "")
    
    ax.set_title(f"Target Count: {target_n} | Detected: {len(detected)}")
    ax.set_xlabel("Raman shift (cm$^{-1}$)")
    ax.set_xlim([0, 800])

axes[0].set_ylabel("Intensity (a.u.)")
axes[0].legend(loc="upper right")
plt.suptitle("FCN Peak Localization: Blue Solid = Ground Truth, Red Dashed = Detected", y=1.02)
plt.tight_layout()
plt.show()

## 4. Stress Test 1: Sandwiched Clusters
Tests detection robustness when low-amplitude peaks are closely flanked by large-amplitude neighbors (a common spectroscopic challenge in crowded fingerprint regions).

In [ ]:
n_trials = 4
fig, axes = plt.subplots(1, n_trials, figsize=(20, 4))

for ax, trial in zip(axes, range(n_trials)):
    true_p = generate_sandwiched_cluster(
        rng, cfg.amp_range, cfg.gamma_range,
        pos_range=(200, 401), window_width_range=(60, 100)
    )
    raw = convert_peaks_to_signal(true_p, cfg.W, peak_type='lorentzian')
    detected = predict_peaks_dense(cnn_model, raw, cfg=cfg, device=device)

    ax.plot(cfg.W, raw, ".", ms=2, color="gray", label="Measurement")
    for A, pos, gamma in true_p:
        ax.axvline(pos, color="#2b6cb0", alpha=0.4, lw=1.5)
    for conf, A, pos, gamma in detected:
        ax.axvline(pos, color="#e53e3e", ls="--", lw=1.2)
    
    ax.set_title(f"Trial {trial+1}: True={len(true_p)}, Detected={len(detected)}")
    ax.set_xlabel("Raman shift (cm$^{-1}$)")
    ax.set_xlim([100, 500])

axes[0].set_ylabel("Intensity (a.u.)")
axes[0].legend(["Measurement"], loc="upper right")
plt.suptitle("Stress Test 1: Sandwiched Peak Clusters (Blue = True, Red Dashed = Detected)", y=1.02)
plt.tight_layout()
plt.show()

## 5. Stress Test 2: Doublet Separation & Resolution Limit
Evaluates the physical resolution limit as the gap between two peaks shrinks relative to their linewidths $\gamma$.

In [ ]:
separations = [3.0, 7.0, 10.0]  # cm^-1 separation
fig, axes = plt.subplots(1, len(separations), figsize=(15, 4))

for ax, sep in zip(axes, separations):
    # Fix linewidth gamma close to separation to test borderline resolution
    true_peaks = generate_close_peaks_sample(
        rng, [sep], amp_range=(2.0, 2.0), gamma_range=None, center=300.0, gamma_fixed=max(sep - 1.5, 1.0)
    )
    raw = convert_peaks_to_signal(true_peaks, W, peak_type='lorentzian')
    detected = predict_peaks_dense(cnn_model, raw, cfg=cfg, device=device)

    ax.plot(W, raw, "-", lw=1.0, color="gray", label="Signal")
    for A, pos, gamma in true_peaks:
        ax.axvline(pos, color="#2b6cb0", alpha=0.4, lw=1.5)
    for conf, A, pos, gamma in detected:
        ax.axvline(pos, color="#e53e3e", ls="--", lw=1.2)
    
    ax.set_title(f"Sep = {sep} cm$^{{-1}}$ | True = 2, Detected = {len(detected)}")
    ax.set_xlabel("Raman shift (cm$^{-1}$)")
    ax.set_xlim([200, 400])

axes[0].set_ylabel("Intensity (a.u.)")
plt.suptitle("Stress Test 2: Close Doublet Separation (Blue = True, Red Dashed = Detected)", y=1.02)
plt.tight_layout()
plt.show()

## 6. Inspecting Failure Cases from Batch Testing
Visualizes the first few profiles where detected peak counts deviated from ground truth.

In [ ]:
n_display = min(4, len(bad_pred))

if n_display > 0:
    fig, axes = plt.subplots(1, n_display, figsize=(18, 3.5))
    if n_display == 1:
        axes = [axes]

    for idx, ax in enumerate(axes):
        raw = bad_raw[idx]
        true_p = bad_peaks[idx]
        detected = bad_pred[idx]

        ax.plot(W, raw, ".", ms=2, color="gray", label="Measurement")
        for A, pos, gamma in true_p:
            ax.axvline(pos, color="#2b6cb0", alpha=0.4, lw=1.5)
        for conf, A, pos, gamma in detected:
            ax.axvline(pos, color="#e53e3e", ls="--", lw=1.2)
        
        ax.set_title(f"Fail #{idx+1}: True={len(true_p)}, Detected={len(detected)}")
        ax.set_xlabel("Raman shift (cm$^{-1}$)")
        ax.set_xlim([0, 800])

    axes[0].set_ylabel("Intensity (a.u.)")
    plt.suptitle("Diagnostic Inspection of Failed Spectra (Blue = True, Red Dashed = Detected)", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No failure profiles detected in batch test.")